In [3]:
!pip install qiskit qiskit-aer pylatexenc

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 30.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.2/14.2 MB 75.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 76.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [qiskit-aer]


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector, Operator
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

sim = AerSimulator()

ЗАДАНИЕ 1

In [ ]:
def full_adder_5q(a_val, b_val, x_val, measure=True):
    qc = QuantumCircuit(5, 2 if measure else 0)
    if a_val: qc.x(0)
    if b_val: qc.x(1)
    if x_val: qc.x(2)

    qc.ccx(0, 1, 4)
    qc.ccx(0, 2, 4)
    qc.ccx(1, 2, 4)
    qc.cx(0, 3)
    qc.cx(1, 3)
    qc.cx(2, 3)

    if measure:
        qc.measure([3, 4], [0, 1])
    return qc

qc_test = full_adder_5q(1, 0, 1, measure=False)
sv = Statevector.from_instruction(qc_test)
print("Вектор состояния (ненулевые амплитуды):", sv.probabilities_dict())

counts = sim.run(full_adder_5q(1, 0, 1, measure=True), shots=1024).result().get_counts()
print("Распределение (CS):", counts)

rows = []
for a in [0, 1]:
    for b in [0, 1]:
        for x in [0, 1]:
            exp_s = a ^ b ^ x
            exp_c = (a & b) | (a & x) | (b & x)
            res = sim.run(full_adder_5q(a, b, x), shots=100).result().get_counts()
            bitstr = max(res, key=res.get)
            got_c, got_s = int(bitstr[0]), int(bitstr[1])
            assert got_s == exp_s and got_c == exp_c, f"Ошибка на {a,b,x}"
            rows.append([a, b, x, exp_s, exp_c, got_s, got_c])

print(pd.DataFrame(rows, columns=["A", "B", "X", "Exp_S", "Exp_C", "Got_S", "Got_C"]))

def full_adder_half(a_val, b_val, x_val):
    qc = QuantumCircuit(8, 2)
    if a_val: qc.x(0)
    if b_val: qc.x(1)
    if x_val: qc.x(2)
    qc.cx(0, 3); qc.cx(1, 3); qc.ccx(0, 1, 4)
    qc.cx(3, 5); qc.cx(2, 5); qc.ccx(3, 2, 6)
    qc.x(4); qc.x(6); qc.ccx(4, 6, 7); qc.x(7); qc.x(4); qc.x(6)
    qc.measure([5, 7], [0, 1])
    return qc

qc5 = full_adder_5q(0, 0, 0, measure=False)
qc8 = full_adder_half(0, 0, 0)
print(f"5q: qubits={qc5.num_qubits}, depth={qc5.depth()}, ops={qc5.count_ops()}")
print(f"Half+OR: qubits={qc8.num_qubits}, depth={qc8.depth()}, ops={qc8.count_ops()}")

def n_bit_adder(n, a_int, b_int):
    qc = QuantumCircuit(4 * n + 1, n + 1)
    for i in range(n):
        if (a_int >> i) & 1: qc.x(i)
        if (b_int >> i) & 1: qc.x(n + i)
    for i in range(n):
        a, b, s = i, n + i, 2 * n + i
        cin, cout = 3 * n + i, 3 * n + i + 1
        qc.ccx(a, b, cout)
        qc.ccx(a, cin, cout)
        qc.ccx(b, cin, cout)
        qc.cx(a, s); qc.cx(b, s); qc.cx(cin, s)
    meas_qubits = [2 * n + i for i in range(n)] + [4 * n]
    qc.measure(meas_qubits, list(range(n + 1)))
    return qc

for a in range(16):
    for b in range(16):
        res = sim.run(n_bit_adder(4, a, b), shots=1).result().get_counts()
        got = int(max(res, key=res.get), 2)
        assert got == a + b, f"Ошибка: {a}+{b} != {got}"
print("Все 256 пар 4-битного сумматора пройдены!")

for bits in [2, 3, 4]:
    qc_n = n_bit_adder(bits, 0, 0)
    print(f"{bits}-bit: qubits={qc_n.num_qubits}, depth={qc_n.depth()}, ops={qc_n.count_ops()}")

ЗАДАНИЕ 2

In [ ]:
def q_and(qc, a, b, target):
    qc.ccx(a, b, target)

def q_or(qc, a, b, target):
    qc.x(a); qc.x(b)
    qc.ccx(a, b, target)
    qc.x(target)
    qc.x(a); qc.x(b)

def q_nand(qc, a, b, target):
    qc.ccx(a, b, target)
    qc.x(target)

def F_classic(a, b, c, d):
    return int(((a or d) and (b or c) and (not (a and c))))

def build_var9_basic():
    comp = QuantumCircuit(10, name="compute_basic")
    q_or(comp, 0, 3, 5)
    q_or(comp, 1, 2, 6)
    q_nand(comp, 0, 2, 7)
    q_and(comp, 5, 6, 8)
    q_and(comp, 8, 7, 9)

    qc = QuantumCircuit(10)
    qc.append(comp, range(10))
    qc.cx(9, 4)
    qc.append(comp.inverse(), range(10))
    return qc.decompose()

def build_var9_opt():
    comp = QuantumCircuit(8, name="compute_opt")
    q_or(comp, 0, 3, 5)
    q_or(comp, 1, 2, 6)
    q_nand(comp, 0, 2, 7)

    qc = QuantumCircuit(8)
    qc.append(comp, range(8))
    qc.mcx([5, 6, 7], 4)
    qc.append(comp.inverse(), range(8))
    return qc.decompose()

def test_circuit(builder, n_ancilla):
    rows = []
    n_q = 5 + n_ancilla
    for idx in range(16):
        a, b, c, d = (idx >> 3) & 1, (idx >> 2) & 1, (idx >> 1) & 1, idx & 1
        qc = QuantumCircuit(n_q, 1 + n_ancilla)
        if a: qc.x(0)
        if b: qc.x(1)
        if c: qc.x(2)
        if d: qc.x(3)
        qc.compose(builder(), inplace=True)
        qc.measure([4] + list(range(5, n_q)), list(range(1 + n_ancilla)))
        counts = sim.run(qc, shots=100).result().get_counts()

        exp_f = F_classic(a, b, c, d)
        exp_str = "0" * n_ancilla + str(exp_f)
        ok = (counts.get(exp_str, 0) == 100)
        assert ok, f"Ошибка на {a,b,c,d}: {counts}"
        rows.append([a, b, c, d, exp_f, int(max(counts, key=counts.get)[-1]), "OK"])
    return pd.DataFrame(rows, columns=["A", "B", "C", "D", "F_class", "F_quant", "Status"])

df_basic = test_circuit(build_var9_basic, 5)
df_opt = test_circuit(build_var9_opt, 3)
print(df_opt)

for name, qc in [("Базовая", build_var9_basic()), ("Оптимизированная", build_var9_opt())]:
    print(f"{name}: qubits={qc.num_qubits}, depth={qc.depth()}, ops={qc.count_ops()}")

ЗАДАНИЕ 3

In [ ]:
p2_values = [0.0, 0.001, 0.005, 0.01, 0.02]
basis_gates = ['u', 'cx']

def eval_with_noise(builder, p2_list, shots=256):
    accs = []
    sample_t = transpile(builder(), basis_gates=basis_gates)
    n_cx = sample_t.count_ops().get('cx', 0)

    for p2 in p2_list:
        noise_model = NoiseModel()
        noise_model.add_all_qubit_quantum_error(depolarizing_error(0.001, 1), ['u'])
        if p2 > 0:
            noise_model.add_all_qubit_quantum_error(depolarizing_error(p2, 2), ['cx'])
        noisy_sim = AerSimulator(noise_model=noise_model)

        correct_total = 0
        for idx in range(16):
            a, b, c, d = (idx >> 3) & 1, (idx >> 2) & 1, (idx >> 1) & 1, idx & 1
            qc = QuantumCircuit(builder().num_qubits, 1)
            if a: qc.x(0)
            if b: qc.x(1)
            if c: qc.x(2)
            if d: qc.x(3)
            qc.compose(builder(), inplace=True)
            qc.measure(4, 0)
            t_qc = transpile(qc, basis_gates=basis_gates)
            counts = noisy_sim.run(t_qc, shots=shots).result().get_counts()
            correct_total += counts.get(str(F_classic(a, b, c, d)), 0)
        accs.append(correct_total / (16 * shots))
    return accs, n_cx

acc_opt, cx_opt = eval_with_noise(build_var9_opt, p2_values)
acc_bas, cx_bas = eval_with_noise(build_var9_basic, p2_values)

plt.plot(p2_values, acc_opt, marker='o', label=f'Оптимизированная (CX={cx_opt})')
plt.plot(p2_values, acc_bas, marker='s', label=f'Базовая (CX={cx_bas})')
plt.xlabel('$p_2$ (шум на CX)')
plt.ylabel('Доля правильных ответов')
plt.legend(); plt.grid(True); plt.show()

ЗАДАНИЕ 4

In [ ]:
def build_oracle(n, marked):
    qc = QuantumCircuit(n, name="Oracle")
    for i, bit in enumerate(reversed(marked)):
        if bit == '0':
            qc.x(i)
    qc.h(n - 1)
    qc.mcx(list(range(n - 1)), n - 1)
    qc.h(n - 1)
    for i, bit in enumerate(reversed(marked)):
        if bit == '0':
            qc.x(i)
    return qc

def build_diffuser(n):
    qc = QuantumCircuit(n, name="Diffuser")
    qc.h(range(n))
    qc.x(range(n))
    qc.h(n - 1)
    qc.mcx(list(range(n - 1)), n - 1)
    qc.h(n - 1)
    qc.x(range(n))
    qc.h(range(n))
    return qc

def build_grover(n, marked_list, k, measure=True):
    if isinstance(marked_list, str):
        marked_list = [marked_list]
    qc = QuantumCircuit(n, n if measure else 0)
    qc.h(range(n))
    diff = build_diffuser(n)
    oracles = [build_oracle(n, m) for m in marked_list]
    for _ in range(k):
        for o in oracles:
            qc.compose(o, inplace=True)
        qc.compose(diff, inplace=True)
    if measure:
        qc.measure(range(n), range(n))
    return qc

for n in [2, 3, 4, 5]:
    target = "10" * (n // 2) + ("1" if n % 2 else "")
    op = Operator(build_oracle(n, target)).data
    diag = np.real(np.diag(op))
    idx_target = int(target, 2)
    expected = np.ones(2**n); expected[idx_target] = -1.0
    assert np.allclose(diag, expected), f"Ошибка оракула при n={n}"
print("Оракул проверен для n=2,3,4,5!")

n = 3
D_mat = Operator(build_diffuser(n)).data
s = np.ones((2**n, 1)) / np.sqrt(2**n)
D_theor = 2 * (s @ s.T) - np.eye(2**n)
assert np.allclose(D_mat, -D_theor) or np.allclose(D_mat, D_theor)
print("Диффузор совпадает с 2|s><s| - I с точностью до глобальной фазы -1!")

theta = np.arcsin(1 / np.sqrt(8))
for k in [1, 2]:
    sv = Statevector.from_instruction(build_grover(3, "101", k, measure=False))
    p_exact = sv.probabilities_dict().get("101", 0)
    p_theor = np.sin((2 * k + 1) * theta)**2
    assert np.isclose(p_exact, p_theor)
    print(f"k={k}: P_exact={p_exact:.6f}, P_theor={p_theor:.6f}")

ЗАДАНИЕ 5

In [ ]:
w5 = "1010011"
n5 = len(w5)
N5 = 2**n5
theta5 = np.arcsin(1 / np.sqrt(N5))
ks = list(range(16))

p_exact_5, p_exp_5, p_theor_5 = [], [], []
for k in ks:
    sv = Statevector.from_instruction(build_grover(n5, w5, k, measure=False))
    p_exact_5.append(sv.probabilities_dict().get(w5, 0))
    p_theor_5.append(np.sin((2 * k + 1) * theta5)**2)
    counts = sim.run(build_grover(n5, w5, k, measure=True), shots=1024).result().get_counts()
    p_exp_5.append(counts.get(w5, 0) / 1024)

plt.plot(ks, p_theor_5, '--', label='Теория')
plt.plot(ks, p_exact_5, 'o-', label='Точная (Statevector)')
plt.plot(ks, p_exp_5, 'x-', label='Эксп. (1024 shots)')
plt.xlabel('k'); plt.ylabel('P(w)'); plt.legend(); plt.grid(True); plt.show()

k_opt_theor = int(np.argmax(p_exact_5))
print(f"k_opt={k_opt_theor}, round(pi/4*sqrt(N))={round(np.pi/4*np.sqrt(N5))}, floor={int(np.floor(np.pi/4*np.sqrt(N5)))}")

shots_list = [16, 64, 256, 1024]
hit_rates = []
for sh in shots_list:
    hits = 0
    for seed in range(20):
        probs = []
        for k in ks:
            c = sim.run(build_grover(n5, w5, k, True), shots=sh, seed_simulator=seed).result().get_counts()
            probs.append(c.get(w5, 0) / sh)
        if int(np.argmax(probs)) == k_opt_theor:
            hits += 1
    hit_rates.append(hits / 20)

plt.plot(shots_list, hit_rates, marker='o')
plt.xscale('log', base=2)
plt.xlabel('shots'); plt.ylabel('Доля совпадений с k_opt'); plt.grid(True); plt.show()

ЗАДАНИЕ 6

In [ ]:
rng = np.random.default_rng(42)
n6 = 6
N6 = 2**n6
all_states = [format(i, f'0{n6}b') for i in range(N6)]

rows6 = []
for M in [1, 2, 3, 4, 8]:
    marked = list(rng.choice(all_states, size=M, replace=False))
    probs = []
    for k in range(16):
        sv = Statevector.from_instruction(build_grover(n6, marked, k, measure=False))
        p_dict = sv.probabilities_dict()
        probs.append(sum(p_dict.get(m, 0) for m in marked))
    plt.plot(range(16), probs, label=f'M={M}')
    k_first = next(i for i in range(1, 15) if probs[i] >= probs[i-1] and probs[i] >= probs[i+1])
    k_theor = int(np.floor(np.pi / 4 * np.sqrt(N6 / M)))
    rows6.append([M, k_theor, k_first, round(probs[k_first], 4)])

plt.xlabel('k'); plt.ylabel('P(k)'); plt.legend(); plt.grid(True); plt.show()
print(pd.DataFrame(rows6, columns=["M", "k_theor", "k_first_max", "P_max"]))

for frac, M in [("N/2", N6 // 2), ("3N/4", 3 * N6 // 4)]:
    marked = list(rng.choice(all_states, size=M, replace=False))
    probs = [sum(Statevector.from_instruction(build_grover(n6, marked, k, False)).probabilities_dict().get(m, 0) for m in marked) for k in range(6)]
    print(f"M={frac} ({M}): P(k=0..5) = {[round(x, 3) for x in probs]}")

ЗАДАНИЕ 7

In [ ]:
w7 = "10110"
n7 = 5
N7 = 2**n7
ks = list(range(16))

p_ex_7, p_ep_7 = [], []
for k in ks:
    sv = Statevector.from_instruction(build_grover(n7, w7, k, measure=False))
    p_ex_7.append(sv.probabilities_dict().get(w7, 0))
    c = sim.run(build_grover(n7, w7, k, measure=True), shots=1024).result().get_counts()
    p_ep_7.append(c.get(w7, 0) / 1024)

k_theor_7 = int(np.floor(np.pi / 4 * np.sqrt(N7)))
k_exact_7 = next(i for i in range(1, 15) if p_ex_7[i] >= p_ex_7[i-1] and p_ex_7[i] >= p_ex_7[i+1])
k_exp_7 = int(np.argmax(p_ep_7[:8]))

print(f"Вариант 9 | N={N7} | |w>=|{w7}> | k_теор={k_theor_7} | k_точн={k_exact_7} | k_эксп={k_exp_7} | max P={p_ex_7[k_exact_7]:.4f}")

plt.plot(ks, p_ex_7, 'k--', label='Идеальная (без шума)')
for p2 in [0.005, 0.02]:
    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(depolarizing_error(p2, 2), ['cx'])
    nsim = AerSimulator(noise_model=nm)
    p_noisy = []
    for k in ks:
        t_qc = transpile(build_grover(n7, w7, k, True), basis_gates=['u', 'cx'])
        c = nsim.run(t_qc, shots=1024).result().get_counts()
        p_noisy.append(c.get(w7, 0) / 1024)
    plt.plot(ks, p_noisy, marker='o', label=f'Шум p2={p2}')

plt.xlabel('k'); plt.ylabel('P(w)'); plt.legend(); plt.grid(True); plt.show()